In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import time
import torch.nn.functional as F
import timm
from sklearn.metrics import accuracy_score, classification_report


IMAGE_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


test_ds      = datasets.ImageFolder("test", transform=transform)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


num_classes = len(test_ds.classes)

model_mnv2 = models.mobilenet_v2(pretrained=False)
model_mnv2.classifier[1] = nn.Linear(
    model_mnv2.classifier[1].in_features,
    num_classes
)

model_mnv2.load_state_dict(torch.load("models/mobilenet_v2.pth", map_location=device))
model_mnv2 = model_mnv2.to(device)
model_mnv2.eval()



predictionresult = []

def inference(model1,loader):
    y_true, y_pred = [], []
    class_names = loader.dataset.classes  
    global_index = 0  

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            out1 = model1(imgs)
            prob1 = F.softmax(out1, dim=1)
            preds = prob1.argmax(dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    return y_true, y_pred


y_true, y_pred = inference(
    model_mnv2,
    test_loader
)
acc = accuracy_score(y_true, y_pred)
print("Ensemble Accuracy:", acc)

print(classification_report(
    y_true,
    y_pred,
    target_names=test_ds.classes
))

